In [1]:
import htcondor
from htcondor import dags
import os
import time
from utils.folder_setup import folder_setup

if not os.path.isdir("logs"):
    os.makedirs("logs/main")
    os.makedirs("logs/null")
    os.makedirs("logs/inference")
    
if not os.path.isdir("Results"):
    os.makedirs("Results/tmp")
    
file = "simulation_data.hdf5"
num_studies = 20
num_true = 5
rep = 1
parameter_test = False
clust_thresh = 0.001

# monte carlo iterations
total = 100
iter_ = total//10

schedd = htcondor.Schedd()

In [2]:
main_wrap = htcondor.Submit({
    "executable": "$ENV(HOME)/projects/pyALE/me_wrap.sh",  # the program to run on the execute node
    "output": f"logs/main/{num_studies}_{num_true}_{rep}.out",       # anything the job prints to standard output will end up in this file
    "error": f"logs/main/{num_studies}_{num_true}_{rep}.err",        # anything the job prints to standard error will end up in this file
    "log": f"logs/main/{num_studies}_{num_true}_{rep}.log",          # this file will contain a record of what happened to the job
    "request_cpus": "1",            # how many CPU cores we want
    "request_memory": "5G",     # how much memory we want
    "arguments": f"{file} {num_studies} {num_true} {rep} -pt {parameter_test}"
})
with schedd.transaction() as txn:
    main_id = main_wrap.queue(txn)

In [3]:
null_wrap = htcondor.Submit({
    "executable": "$ENV(HOME)/projects/pyALE/null_wrap.sh",  # the program to run on the execute node
    "output": f"logs/null/{num_studies}_{num_true}_{rep}_$(ProcId).out",       # anything the job prints to standard output will end up in this file
    "error": f"logs/null/{num_studies}_{num_true}_{rep}_$(ProcId).err",        # anything the job prints to standard error will end up in this file
    "log": f"logs/null/{num_studies}_{num_true}_{rep}_$(ProcId).log",          # this file will contain a record of what happened to the job
    "request_cpus": "1",            # how many CPU cores we want
    "request_memory": "2G",      # how much memory we want
    "arguments": f"{clust_thresh} {num_studies} {num_true} {rep} {iter_}"
})


with schedd.transaction() as txn:
    null_id = null_wrap.queue(txn, count=total//iter_)

In [4]:
inference_wrap = htcondor.Submit({
    "executable": "$ENV(HOME)/projects/pyALE/inference_wrap.sh",  # the program to run on the execute node
    "output": f"logs/inference/{num_studies}_{num_true}_{rep}.out",       # anything the job prints to standard output will end up in this file
    "error": f"logs/inference/{num_studies}_{num_true}_{rep}.err",        # anything the job prints to standard error will end up in this file
    "log": f"logs/inference/{num_studies}_{num_true}_{rep}.log",          # this file will contain a record of what happened to the job
    "request_cpus": "1",            # how many CPU cores we want
    "request_memory": "2G",      # how much memory we want
    "arguments": f"{clust_thresh} {num_studies} {num_true} {rep} {iter_}"
})


with schedd.transaction() as txn:
    inference_id = inference_wrap.queue(txn)